# Proyecto #1: Biodiversity at Scale
## Parte 7: Biodiversity Long-Tail Challenge
**Maestría de Investigación en IA - UTEC Posgrado**  
**Curso**: Aprendizaje Profundo – Práctica  
**Profesor**: Dra. Aurea Soriano-Vargas

---
### Objetivos de esta etapa:
1. Construir un escenario experimental **Long-Tail sintético y reproducible** (`seed=42`) con clases:
   - **Frecuentes (*Many-shot*)**: 35 muestras/clase (10 especies).
   - **Intermedias (*Medium-shot*)**: 15 muestras/clase (20 especies).
   - **Minoritarias (*Few-shot*)**: 3 muestras/clase (20 especies).
2. Entrenar el modelo base bajo distribución desbalanceada (**E10: Long-Tail sin corrección**).
3. Evaluar y comparar una estrategia de mitigación (**E11: Long-Tail con Focal Loss**).
4. Reportar métricas desglosadas por nivel de abundancia (Frequent vs Medium vs Minority).


In [ ]:
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))

import torch
from src.utils.seed import seed_everything
from src.data.dataset import SyntheticINatDataset
from src.data.sampler import create_long_tail_subset, get_class_balanced_weights
from src.data.dataloader import build_dataloaders
from src.models.factory import build_model, count_parameters
from src.training.losses import build_criterion
from src.training.optimizers import build_optimizer
from src.training.trainer import Trainer
from src.utils.tracking import ExperimentTracker
from src.utils.visualization import plot_long_tail_comparison

SEED = 42
seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tracker = ExperimentTracker(log_dir=str(ROOT_DIR / "logs"))


### 1. Construcción del Split Long-Tail Reproducible


In [ ]:
N_CLASSES = 50
train_base = SyntheticINatDataset(num_samples=N_CLASSES * 35, num_classes=N_CLASSES, img_size=128, seed=SEED)
val_ds = SyntheticINatDataset(num_samples=N_CLASSES * 10, num_classes=N_CLASSES, img_size=128, seed=SEED+1)

selected_classes = list(range(N_CLASSES))
lt_indices, lt_map, class_tiers = create_long_tail_subset(
    train_base,
    selected_classes=selected_classes,
    many_shot_count=35,
    med_shot_count=15,
    few_shot_count=3,
    many_ratio=0.20,
    med_ratio=0.40,
    seed=SEED,
    output_manifest=str(ROOT_DIR / "configs" / "long_tail_manifest_seed42.json")
)

from torch.utils.data import Subset
train_lt = Subset(train_base, lt_indices)

print(f"Total muestras Long-Tail Train: {len(train_lt)}")
print(f"  - Clases Frecuentes (Many-shot): {len(class_tiers['frequent'])} especies")
print(f"  - Clases Intermedias (Med-shot): {len(class_tiers['medium'])} especies")
print(f"  - Clases Minoritarias (Few-shot): {len(class_tiers['minority'])} especies")


### 2. Experimento E10: Long-Tail Sin Corrección (Cross-Entropy Estándar)


In [ ]:
train_loader, val_loader = build_dataloaders(train_lt, val_ds, batch_size=32, num_workers=2, seed=SEED)
ce_loss = build_criterion("cross_entropy")

model_e10 = build_model("resnet18", num_classes=N_CLASSES, mode="partial_fine_tuning", pretrained=True)
opt_e10 = build_optimizer(model_e10, opt_type="adamw", lr=1e-3)

trainer_e10 = Trainer(model_e10, ce_loss, opt_e10, device=device, class_tiers=class_tiers, use_amp=True)
hist_e10, best_e10, vram_e10, time_e10 = trainer_e10.fit(train_loader, val_loader, epochs=4, verbose=True)

tracker.log_experiment("E10", "ResNet-18 (Partial FT)", "AdamW", "BN+Dropout", "Standard", "Partial FT", "Long-Tail (Sin Mitigación)", best_e10, time_e10, vram_e10, 11.2)


### 3. Experimento E11: Long-Tail Con Mitigación (Focal Loss)
Focal Loss repondera dinámicamente los ejemplos fáciles y penaliza severamente los errores en las clases con baja probabilidad a priori.


In [ ]:
focal_loss = build_criterion("focal_loss", gamma=2.0)

model_e11 = build_model("resnet18", num_classes=N_CLASSES, mode="partial_fine_tuning", pretrained=True)
opt_e11 = build_optimizer(model_e11, opt_type="adamw", lr=1e-3)

trainer_e11 = Trainer(model_e11, focal_loss, opt_e11, device=device, class_tiers=class_tiers, use_amp=True)
hist_e11, best_e11, vram_e11, time_e11 = trainer_e11.fit(train_loader, val_loader, epochs=4, verbose=True)

tracker.log_experiment("E11", "ResNet-18 (Partial FT)", "AdamW", "BN+Dropout", "Standard", "Partial FT", "Long-Tail (Focal Loss)", best_e11, time_e11, vram_e11, 11.2)


### 4. Comparación de Rendimiento por Niveles de Abundancia


In [ ]:
plot_long_tail_comparison(
    baseline_metrics=best_e10,
    mitigated_metrics=best_e11,
    title="Impacto de Focal Loss en Clases Frecuentes vs. Minoritarias"
)
